# Milestone 6: Demand Forecasting

## 6.1. Import Necessary Libraries and Load Cleaned Datasets

In [1]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

In [2]:


# Load cleaned datasets
orders = pd.read_csv("../data/processed/orders.csv", parse_dates=["order_date"])
items = pd.read_csv("../data/processed/order_items.csv")

# Connect each purchased item to its order date and store
sales = items.merge(
    orders[["order_id", "order_date", "store_id"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

# Extract the month of each sale
sales["year_month"] = sales["order_date"].dt.to_period("M").dt.to_timestamp()

# Calculate monthly units sold by each store
monthly = (
    sales.groupby(["store_id", "year_month"], as_index=False)["quantity"]
    .sum()
    .rename(columns={"quantity": "units_sold"})
    .sort_values(["store_id", "year_month"])
)

# Check our results
print("Dataset shape:", monthly.shape)
print("Date range:", monthly["year_month"].min(), "to", monthly["year_month"].max())

display(monthly.head())

Dataset shape: (249, 3)
Date range: 2024-01-01 00:00:00 to 2026-09-01 00:00:00


,store_id,year_month,units_sold
0,S001,2024-01-01,5444
1,S001,2024-02-01,4474
2,S001,2024-03-01,5099
3,S001,2024-04-01,4590
4,S001,2024-05-01,5056


## 6.2 Check Monthly Data Coverage

In [3]:
# Check the number of months available for each store
store_coverage = monthly.groupby("store_id").agg(
    months=("year_month", "count"),
    first_month=("year_month", "min"),
    last_month=("year_month", "max")
)

display(store_coverage)

# Confirm that aggregation preserved all units sold
print("Original total units:", items["quantity"].sum())
print("Monthly total units:", monthly["units_sold"].sum())

,months,first_month,last_month
store_id,,,
S001,33,2024-01-01,2026-09-01
S002,33,2024-01-01,2026-09-01
S003,33,2024-01-01,2026-09-01
S004,33,2024-01-01,2026-09-01
S005,33,2024-01-01,2026-09-01
S006,33,2024-01-01,2026-09-01
S007,20,2025-02-01,2026-09-01
S008,16,2025-06-01,2026-09-01
S009,9,2026-01-01,2026-09-01


Original total units: 1227235
Monthly total units: 1227235


## 6.3 Feature Engineering

In [4]:
# Sort data by store and month
monthly = monthly.sort_values(["store_id", "year_month"]).copy()

# Create historical sales features
monthly["lag_1"] = monthly.groupby("store_id")["units_sold"].shift(1)
monthly["lag_2"] = monthly.groupby("store_id")["units_sold"].shift(2)
monthly["lag_3"] = monthly.groupby("store_id")["units_sold"].shift(3)

# Extract calendar month to capture seasonality
monthly["month"] = monthly["year_month"].dt.month

# Remove rows without three months of history
model_data = monthly.dropna().copy()

# Check results
print("Model dataset shape:", model_data.shape)
display(model_data.head())

Model dataset shape: (219, 7)


,store_id,year_month,units_sold,lag_1,lag_2,lag_3,month
3,S001,2024-04-01,4590,5099.0,4474.0,5444.0,4
4,S001,2024-05-01,5056,4590.0,5099.0,4474.0,5
5,S001,2024-06-01,5185,5056.0,4590.0,5099.0,6
6,S001,2024-07-01,5070,5185.0,5056.0,4590.0,7
7,S001,2024-08-01,5652,5070.0,5185.0,5056.0,8


In [5]:
# Split data by time
train = model_data[model_data["year_month"] < "2026-04-01"].copy()

val = model_data[
    (model_data["year_month"] >= "2026-04-01") &
    (model_data["year_month"] < "2026-07-01")
].copy()

test = model_data[model_data["year_month"] >= "2026-07-01"].copy()

# Check the splits
print("Training:", train.shape)
print("Validation:", val.shape)
print("Testing:", test.shape)

Training: (162, 7)
Validation: (27, 7)
Testing: (30, 7)


## 6.4 Baseline Model

In [6]:
# Actual sales and baseline predictions
actual = val["units_sold"]
baseline = val["lag_1"]

# Calculate errors
baseline_mae = mean_absolute_error(actual, baseline)
baseline_rmse = mean_squared_error(actual, baseline) ** 0.5

print("Baseline MAE:", round(baseline_mae, 2))
print("Baseline RMSE:", round(baseline_rmse, 2))

Baseline MAE: 258.78
Baseline RMSE: 293.95


## 6.5 Train Model

In [7]:
# Select features
features = ["lag_1", "lag_2", "lag_3", "month"]

# Create and train the model
model = RandomForestRegressor(n_estimators=100, random_state=42)

model.fit(train[features], train["units_sold"])

# Predict validation sales
predictions = model.predict(val[features])

# Evaluate the model
rf_mae = mean_absolute_error(val["units_sold"], predictions)
rf_rmse = mean_squared_error(val["units_sold"], predictions) ** 0.5

print("Random Forest MAE:", round(rf_mae, 2))
print("Random Forest RMSE:", round(rf_rmse, 2))

print("\nBaseline MAE:", round(baseline_mae, 2))
print("Baseline RMSE:", round(baseline_rmse, 2))

Random Forest MAE: 219.06
Random Forest RMSE: 256.4

Baseline MAE: 258.78
Baseline RMSE: 293.95


## 6.6 Final Model Evaluation

In [8]:
# Combine training and validation data
train_val = pd.concat([train, val])

# Train the final model
final_model = RandomForestRegressor(n_estimators=100, random_state=42)

final_model.fit(train_val[features], train_val["units_sold"])

# Predict test data
test_predictions = final_model.predict(test[features])

# Calculate baseline errors
test_baseline_mae = mean_absolute_error(test["units_sold"], test["lag_1"])
test_baseline_rmse = mean_squared_error(test["units_sold"], test["lag_1"]) ** 0.5

# Calculate Random Forest errors
test_rf_mae = mean_absolute_error(test["units_sold"], test_predictions)
test_rf_rmse = mean_squared_error(test["units_sold"], test_predictions) ** 0.5

# Compare results
print("Baseline MAE:", round(test_baseline_mae, 2))
print("Baseline RMSE:", round(test_baseline_rmse, 2))

print("\nRandom Forest MAE:", round(test_rf_mae, 2))
print("Random Forest RMSE:", round(test_rf_rmse, 2))

Baseline MAE: 455.37
Baseline RMSE: 507.15

Random Forest MAE: 264.55
Random Forest RMSE: 322.3


## 6.7 Prediction Errors

In [9]:
# Compare actual sales with predictions
results = test[["store_id", "year_month", "units_sold"]].copy()

results["predicted_units"] = test_predictions.round(0).astype(int)

# Calculate prediction errors
results["error"] = results["predicted_units"] - results["units_sold"]
results["absolute_error"] = results["error"].abs()

# Show the 10 largest prediction errors
display(results.sort_values("absolute_error", ascending=False).head(10))

# Average error for each store
display(results.groupby("store_id")["absolute_error"].mean().sort_values(ascending=False))

,store_id,year_month,units_sold,predicted_units,error,absolute_error
31,S001,2026-08-01,6400,5643,-757,757
64,S002,2026-08-01,6327,5675,-652,652
32,S001,2026-09-01,5677,5197,-480,480
216,S007,2026-08-01,5260,4794,-466,466
247,S010,2026-08-01,4754,4289,-465,465
241,S009,2026-08-01,5106,4649,-457,457
197,S006,2026-09-01,4479,4891,412,412
65,S002,2026-09-01,5539,5160,-379,379
63,S002,2026-07-01,5745,5372,-373,373
30,S001,2026-07-01,5777,5437,-340,340


store_id
S001    525.666667
S002    468.000000
S010    278.000000
S009    231.666667
S007    230.333333
S006    228.000000
S005    202.666667
S003    179.000000
S004    162.333333
S008    139.333333
Name: absolute_error, dtype: float64

### Prediction Error Analysis

The Random Forest outperformed the baseline, achieving an MAE of 264.55 units compared to 455.37 units.

The largest error occurred at store S001 in August 2026, where the model underestimated sales by 757 units. Stores S001 and S002 had the highest average errors.

The model struggled with some demand increases, particularly in August. More historical data and additional features could improve its accuracy.

## 6.8 October 2026 Demand Forecast

In [10]:
# Train the model using all available data
model.fit(model_data[features], model_data["units_sold"])

# Get the latest month for each store
october = model_data.groupby("store_id").tail(1).copy()

# Update lag features for October
october["lag_3"] = october["lag_2"]
october["lag_2"] = october["lag_1"]
october["lag_1"] = october["units_sold"]

# Set the forecast month
october["year_month"] = pd.Timestamp("2026-10-01")
october["month"] = 10

# Predict October demand
october["forecast_units"] = model.predict(october[features]).round().astype(int)

# Display forecasts
forecast = october[["store_id", "year_month", "forecast_units"]]

display(forecast)

print("Total forecast units:", forecast["forecast_units"].sum())

,store_id,year_month,forecast_units
32,S001,2026-10-01,5280
65,S002,2026-10-01,5309
98,S003,2026-10-01,5001
131,S004,2026-10-01,5334
164,S005,2026-10-01,5393
197,S006,2026-10-01,4876
217,S007,2026-10-01,4945
233,S008,2026-10-01,4586
242,S009,2026-10-01,4866
248,S010,2026-10-01,4546


Total forecast units: 50136


## 6.9 Save Forecast

In [11]:
# Save October 2026 forecast
forecast.to_csv("../data/processed/october_2026_forecast.csv", index=False)

print("Forecast saved successfully.")

Forecast saved successfully.
